In [29]:
import os
import random
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from scipy.optimize import minimize
from sklearn.metrics import log_loss, accuracy_score, precision_recall_fscore_support, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm



In [30]:
# ==========================================
# 1. CONFIGURATION


In [31]:
# ==========================================
class TrainingConfig:
    # System
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Model
    BACKBONE_MODEL = "microsoft/deberta-v3-base"
    HIDDEN_SIZE = 768
    
    # LoRA
    LORA_R = 16
    USE_LORA_STAGE1 = True
    FREEZE_LORA_STAGE2 = True # Option A
    
    # Data
    AUGMENTATION_MAX_PER_ORIGINAL = 8
    OOD_TEST_RATIO = 0.05
    
    # Training Stages
    STAGE1_EPOCHS = 3   # Triplet
    STAGE2_EPOCHS = 5   # Multi-head
    BATCH_SIZE = 16
    MAX_LENGTH = 256
    
    # Optimization
    LR_BACKBONE = 1e-5
    LR_REASONER = 1e-4
    
    # Thresholds
    OOD_F1_THRESHOLD = 0.70
    HOLD_PRECISION_THRESHOLD = 0.85

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

config = TrainingConfig()
set_seed(config.SEED)
print(f"Using device: {config.DEVICE}")



Using device: cuda


In [32]:
# ==========================================
# 2. DATASET CLASSES


In [33]:
# ==========================================
import torch
from torch.utils.data import Dataset
import numpy as np
import pandas as pd

class NexusTripletDataset(Dataset):
    """
    NEXUS v4.5: Original_ID bazlı Causal Triplet Mining.
    Aynı haberin Long ve Short versiyonlarını Hard Negative olarak kullanır.
    """
    def __init__(self, df):
        self.samples = []
        # original_id yoksa gruplama yapılamaz, bu yüzden kontrol ekliyoruz.
        if 'original_id' not in df.columns:
            raise ValueError("Dataset must contain 'original_id' for Causal Triplet Mining.")
            
        self.df = df
        self.grouped = df.groupby('original_id')
        
        for oid, group in self.grouped:
            # Etiketlere göre ayır (0: Hold, 1: Short, 2: Long)
            long_samples = group[group['label'] == 2]
            short_samples = group[group['label'] == 1]
            
            # --- LONG Anchor Senaryosu ---
            for _, anchor_row in long_samples.iterrows():
                # Positive: Aynı ID'ye sahip başka bir Long (Augmentation)
                pos_candidates = long_samples[long_samples.index != anchor_row.name]
                if not pos_candidates.empty:
                    positive = pos_candidates.sample(1).iloc[0]['text']
                else:
                    # Global havuzdan aynı varlığa [C] sahip başka bir Long bul
                    asset = anchor_row['text'].split('[C]')[1].strip() if '[C]' in anchor_row['text'] else ""
                    alt_pos = df[(df['label'] == 2) & (df['text'].str.contains(asset, regex=False)) & (df.index != anchor_row.name)]
                    if not alt_pos.empty:
                        positive = alt_pos.sample(1).iloc[0]['text']
                    else: continue

                # HARD NEGATIVE: Aynı haberin SHORT (Counterfactual) versiyonu
                if not short_samples.empty:
                    negative = short_samples.sample(1).iloc[0]['text']
                    self.samples.append((anchor_row['text'], positive, negative))
                
            # --- SHORT Anchor Senaryosu ---
            for _, anchor_row in short_samples.iterrows():
                pos_candidates = short_samples[short_samples.index != anchor_row.name]
                if not pos_candidates.empty:
                    positive = pos_candidates.sample(1).iloc[0]['text']
                else:
                    asset = anchor_row['text'].split('[C]')[1].strip() if '[C]' in anchor_row['text'] else ""
                    alt_pos = df[(df['label'] == 1) & (df['text'].str.contains(asset, regex=False)) & (df.index != anchor_row.name)]
                    if not alt_pos.empty:
                        positive = alt_pos.sample(1).iloc[0]['text']
                    else: continue

                # HARD NEGATIVE: Aynı haberin LONG (Counterfactual) versiyonu
                if not long_samples.empty:
                    negative = long_samples.sample(1).iloc[0]['text']
                    self.samples.append((anchor_row['text'], positive, negative))

        print(f"✓ Created {len(self.samples)} Causal Triplets from {len(self.grouped)} unique news events.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # Anchor, Positive, Negative metinlerini döner
        return self.samples[idx]

class NexusDatasetV2(Dataset):
    """
    NEXUS v4.5: Multi-Task Dataset.
    Yeni JSON formatındaki tp_pct ve validity_minutes alanlarını işler.
    """
    def __init__(self, df):
        self.texts = df['text'].tolist()
        
        # 1. GATE: Aksiyon var mı? (0: No, 1: Yes)
        self.gates = (df['label'] != 0).astype(float).values
        
        # 2. DIRECTIONS: (HOLD: -1, SHORT: 0, LONG: 1)
        # Loss v7_final bu formatı bekler.
        self.directions = np.where(df['label'] == 0, -1.0, 
                                   (df['label'] == 2).astype(float)).astype(float)
        
        # 3. REGRESSION: TP ve Validity
        self.tps = df['tp_pct'].fillna(0.0).values
        self.validities = df['validity_minutes'].fillna(0.0).values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            self.texts[idx],
            torch.tensor(self.gates[idx], dtype=torch.float32),
            torch.tensor(self.directions[idx], dtype=torch.float32),
            torch.tensor(self.tps[idx], dtype=torch.float32),
            torch.tensor(self.validities[idx], dtype=torch.float32),
        )

In [34]:
# ==========================================
# 3. MODEL ARCHITECTURE


In [ ]:
# D. STAGE 2: MULTI-HEAD TRAINING
print(f"\n=== STAGE 2: MULTI-HEAD TRAINING ({config.STAGE2_EPOCHS} Epochs) ===")

# Freeze logic
"""
if config.FREEZE_LORA_STAGE2:
    print("Freezing Encoder (including LoRA)...")
    for param in model.encoder.parameters():
        param.requires_grad = False
"""
train_ds = NexusDatasetV2(train_df)
val_ds = NexusDatasetV2(val_df)

# Weighted Sampler
train_labels = (train_df['label'] != 0).astype(int).values
class_counts = np.bincount(train_labels)
class_weights = np.sqrt(1.0 / class_counts)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False)

optimizer = get_nexus_optimizer(model)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=len(train_loader), num_training_steps=len(train_loader)*config.STAGE2_EPOCHS)

best_val_loss = float('inf')
patience_counter = 0
for epoch in range(config.STAGE2_EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in pbar:
        texts, gate_gt, dir_gt, tp_gt, val_gt = batch

        gate_gt = gate_gt.view(-1, 1).to(config.DEVICE)
        dir_gt = dir_gt.view(-1, 1).to(config.DEVICE)
        tp_gt = tp_gt.view(-1, 1).to(config.DEVICE)
        val_gt = val_gt.view(-1, 1).to(config.DEVICE)

        inputs = model.tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=config.MAX_LENGTH).to(config.DEVICE)
        outputs = model.forward(inputs['input_ids'], inputs['attention_mask'])

        loss, loss_dict = nexus_combined_loss_v7_final(outputs, (gate_gt, dir_gt, tp_gt, val_gt))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = total_loss / len(train_loader)
    # Validation
    model.eval()
    val_loss = 0
    all_gate_gts = []
    all_gate_preds = []
    all_dir_gts = []
    all_dir_preds = []
    all_tp_gts = []
    all_tp_preds = []
    all_val_gts = []
    all_val_preds = []
    with torch.no_grad():
        for batch in val_loader:
            texts, gate_gt, dir_gt, tp_gt, val_gt = batch
            gate_gt = gate_gt.view(-1, 1).to(config.DEVICE)
            dir_gt = dir_gt.view(-1, 1).to(config.DEVICE)
            tp_gt = tp_gt.view(-1, 1).to(config.DEVICE)
            val_gt = val_gt.view(-1, 1).to(config.DEVICE)

            inputs = model.tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=config.MAX_LENGTH).to(config.DEVICE)
            outputs = model.forward(inputs['input_ids'], inputs['attention_mask'])
            preds = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], gate_labels=gate_gt)
            loss, _ = nexus_combined_loss_v7_final(outputs, (gate_gt, dir_gt, tp_gt, val_gt))
            val_loss += loss.item()

            gate_p = preds['gate']
            dir_p = preds['direction']
            tp_p = preds['tp']
            val_p = preds['validity']

            all_gate_gts.extend(gate_gt.cpu().numpy())
            all_gate_preds.extend(torch.sigmoid(outputs['gate']).cpu().numpy() > 0.5)
            action_mask = (gate_gt == 1).squeeze()
            valid_dir_mask = (dir_gt >= 0).squeeze()  
            combined_mask = action_mask & valid_dir_mask
            if combined_mask.sum() > 0:
                  dir_pred = (torch.sigmoid(dir_p[combined_mask]) > 0.5).float()
                  all_dir_preds.extend(dir_pred.cpu().numpy())
                  all_dir_gts.extend(dir_gt[combined_mask].cpu().numpy())
                  all_tp_preds.extend(tp_p[action_mask].cpu().numpy())
                  all_tp_gts.extend(tp_gt[action_mask].cpu().numpy())
                  all_val_preds.extend(val_p[action_mask].cpu().numpy())
                  all_val_gts.extend(val_gt[action_mask].cpu().numpy())
    avg_val = val_loss / len(val_loader)
    # Metrics
    gate_acc = accuracy_score(all_gate_gts, all_gate_preds)
    gate_prec, gate_rec, gate_f1, _ = precision_recall_fscore_support(
        all_gate_gts, all_gate_preds, average='binary', zero_division=0
    )
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/5 Results")
    print(f"{'='*50}")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val:.4f}")
    print(f"Gate - Acc: {gate_acc:.3f} | Prec: {gate_prec:.3f} | Rec: {gate_rec:.3f} | F1: {gate_f1:.3f}")
    if len(all_dir_preds) > 0:
        dir_acc = accuracy_score(all_dir_gts, all_dir_preds)
        tp_mae = mean_absolute_error(all_tp_gts, all_tp_preds)
        val_mae = mean_absolute_error(all_val_gts, all_val_preds)
        print(f"Direction Acc: {dir_acc:.3f}")
        print(f"TP MAE: {tp_mae:.3}")
        print(f"Validity MAE: {val_mae:.3}")
# Early stopping - SADECE val_loss
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "nexus_best.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val,
            'val_f1': gate_f1
        }, "nexus_checkpoint.pt")
        print(f"✓ Model saved (val_loss: {avg_val:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"✗ No improvement ({patience_counter}/3)")
        if patience_counter >= 3:
            print("Early stopping triggered!")
            break


# E. CALIBRATION & TESTING
model.load_state_dict(torch.load("nexus_production_final.pt"))
optimal_t = calibrate_model(model, val_loader, config.DEVICE)
counterfactual_test(model)
print("\n✓ Training Complete. Model saved as 'nexus_production_final.pt'")



In [36]:
# ==========================================
# 4. LOSS FUNCTIONS


In [ ]:
def nexus_combined_loss_v7_final(outputs, targets, gamma=2.0, label_smoothing=0.1):
    gate_p, dir_p, tp_p, val_p = outputs['gate'], outputs['direction'], outputs['tp'], outputs['validity']
    gate_gt, dir_gt, tp_gt, val_gt = targets

    # Gate Loss: Smoothing aktif (Ezber önleyici)
    gate_gt_s = gate_gt * (1 - label_smoothing) + 0.5 * label_smoothing
    loss_gate = F.binary_cross_entropy_with_logits(gate_p, gate_gt_s, pos_weight=torch.tensor([2.0]).to(gate_p.device))

    # Direction Loss: Pure Focal (Zıtlıkları yakalamak için keskin gradyan)
    valid_mask = (dir_gt >= 0).squeeze()
    if valid_mask.sum() > 0:
        v_p, v_gt = dir_p[valid_mask], dir_gt[valid_mask]
        weights = torch.where(v_gt == 0, 3.0, 1.0) # Short cezası
        probs = torch.sigmoid(v_p)
        pt = torch.where(v_gt == 1, probs, 1 - probs)
        loss_dir = (weights * ((1 - pt) ** gamma) * F.binary_cross_entropy_with_logits(v_p, v_gt, reduction='none')).mean()
    else:
        loss_dir = torch.tensor(0.0, device=gate_p.device)

    # Regression: Huber
    act_mask = (gate_gt == 1).squeeze()
    loss_reg = (F.huber_loss(tp_p[act_mask], tp_gt[act_mask]) + F.huber_loss(val_p[act_mask], val_gt[act_mask])) if act_mask.sum() > 0 else torch.tensor(0.0, device=gate_p.device)

    return (1.0 * loss_gate) + (4.0 * loss_dir) + (0.5 * loss_reg), {'dir_focal': loss_dir.item()}

In [38]:
# ==========================================
# 5. TRAINING FUNCTIONS


In [ ]:
# ==========================================
def apply_lora_final(st_model, r=16):
    """
    GPT Fix: Exclude output.dense to avoid representation collapse.
    """
    # Assuming st_model here refers to the encoder inside NexusV2Production or similar,
    # but based on usage in notebook, it takes the whole model then accesses auto_model.
    # We will adjust to apply to 'model.encoder' directly.
    
    target_model = st_model.encoder
    lora_config = LoraConfig(
        r=r,
        lora_alpha=r*2,
        target_modules=["query_proj", "key_proj", "value_proj", "intermediate.dense"],
        lora_dropout=0.3,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION 
    )
    st_model.encoder = get_peft_model(target_model, lora_config)
    st_model.encoder.print_trainable_parameters()
    return st_model

class CausalTripletTrainer:
    """
    NEXUS v4.6: Stage 1 Causal Trainer.
    Backbone ve Reasoning Block'u senkronize eğitir.
    """
    def __init__(self, model, train_loader, margin=0.4, lr_backbone=1e-5, lr_reasoner=1e-4):
        self.model = model
        self.train_loader = train_loader
        self.device = next(model.parameters()).device
        
        # Triplet Loss: Cosine Distance tabanlı (Margin'i 0.4'e çıkardık!)
        self.criterion = nn.TripletMarginWithDistanceLoss(
            distance_function=lambda x, y: 1.0 - F.cosine_similarity(x, y),
            margin=margin
        )
        
        # GPT-Proof Optimizer: LoRA (Warm) + Reasoner (Hot)
        self.optimizer = AdamW([
            {'params': [p for n, p in model.encoder.named_parameters() if "lora_" in n], 'lr': lr_backbone},
            {'params': model.gate_reasoner.parameters(), 'lr': lr_reasoner}
        ], weight_decay=0.01)

    def train_epoch(self):
        self.model.train()
        total_loss = 0
        pbar = tqdm(self.train_loader, desc="Stage 1: Causal Triplet Training")
        
        for batch in pbar:
            anchors, positives, negatives = batch
            self.optimizer.zero_grad()
            
            # Üç kolu da Reasoning Block üzerinden geçiriyoruz
            a_emb = self._get_embedding(anchors)
            p_emb = self._get_embedding(positives)
            n_emb = self._get_embedding(negatives)
            
            # Loss hesaplama ve geriye yayılım
            loss = self.criterion(a_emb, p_emb, n_emb)
            loss.backward()
            
            # Gradyan Kırpma (Reasoning Block için hayati emniyet kemeri)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            
            self.optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({'triplet_loss': f'{loss.item():.4f}'})
            
        return total_loss / len(self.train_loader)

    def _get_embedding(self, texts):
        """Haber metnini DeepReasoningBlock çıkışındaki context vektörüne dönüştürür."""
        # Tokenization süreci
        inputs = self.model.tokenizer(
            list(texts), return_tensors="pt", padding=True, truncation=True, max_length=256
        ).to(self.device)
        
        # Backbone + Reasoner Akışı (Muhakeme edilmiş vektör üretimi)
        with torch.set_grad_enabled(True): # Stage 1'de gradyanlar açık olmalı
            encoder_out = self.model.encoder(inputs['input_ids'], inputs['attention_mask']).last_hidden_state
            # gate_reasoner artık 2 katmanlı hiyerarşik muhakeme yapıyor
            context = self.model.gate_reasoner(encoder_out, inputs['attention_mask'])
            
        return context

def calibrate_model(model, val_loader, device):
    print("Calibrating Temperature...")
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            texts, gate_gt, _, _, _ = batch
            inputs = model.tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            outputs = model.forward(inputs['input_ids'], inputs['attention_mask'])
            all_logits.append(outputs['gate'].cpu())
            all_labels.append(gate_gt.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()

    def objective(T):
        scaled_probs = 1 / (1 + np.exp(-logits / T))
        return log_loss(labels, scaled_probs)

    res = minimize(objective, x0=[1.0], bounds=[(0.01, 10.0)], method='L-BFGS-B')
    optimal_t = res.x[0]
    print(f"✓ Optimal Temperature Found: {optimal_t:.4f}")
    return optimal_t

def get_nexus_optimizer(model, base_lr=1e-4, lora_lr=1e-5):
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        # Group 1: New Heads & Reasoners (Hot)
        {
            "params": [p for n, p in model.named_parameters() 
                       if ("reasoner" in n or "out" in n or "head" in n) 
                       and not any(nd in n for nd in no_decay)],
            "weight_decay": 0.05,
            "lr": base_lr,
        },
        # Group 2: LoRA Adapters (Warm)
        {
            "params": [p for n, p in model.named_parameters() 
                       if "lora_" in n and not any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": lora_lr,
        },
        # No decay for bias/layernorm
        {
            "params": [p for n, p in model.named_parameters() 
                       if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": base_lr,
        },
    ]
    return AdamW(optimizer_grouped_parameters, eps=1e-8)

def counterfactual_test(model):
    print("\n" + "="*50)
    print("COUNTERFACTUAL CAUSALITY TEST")
    print("="*50)

    test_pairs = [
        ("[N] Bitcoin ETF approved by SEC [C] BTC", "[N] Bitcoin ETF rejected by SEC [C] BTC"),
        ("[N] Ethereum upgrade successful [C] ETH", "[N] Ethereum upgrade delayed [C] ETH"),
        ("[N] Solana gains 15% in morning trading session [C] SOL", "[N] Solana drops 15% in morning trading session [C] SOL"),
        ("[N] Concerns over Polygon network stability prove unfounded [C] MATIC", "[N] Concerns over Polygon network stability prove justified [C] MATIC"),
        ("[N] Bitcoin finally breaks resistance at $70K after months [C] BTC", "[N] Bitcoin fails again to break resistance at $70K [C] BTC"),
        ("[N] Ethereum outperforms Bitcoin in Q4 rally [C] ETH", "[N] Bitcoin outperforms Ethereum in Q4 rally [C] BTC"),
        ("[N] Experts say crypto winter is 'definitely' over this time [C] BTC", "[N] Experts confirm crypto winter continues with no end in sight [C] BTC"),
    ]

    passed = 0
    total = len(test_pairs)

    for orig, counter in test_pairs:
        orig_pred = model.predict(orig)
        counter_pred = model.predict(counter)

        if orig_pred['direction'] != counter_pred['direction']:
            print(f"  ✓ PASS: {orig[:30]}... ({orig_pred['direction']}) -> {counter[:30]}... ({counter_pred['direction']})")
            passed += 1
        else:
            print(f"  ✗ FAIL: {orig[:30]}... ({orig_pred['direction']}) -> {counter[:30]}... ({counter_pred['direction']})")

    print(f"\nResult: {passed}/{total} passed ({passed/total*100:.1f}%)")
    return passed / total



In [40]:
# ==========================================
# 6. MAIN EXECUTION FLOW


In [41]:
# ==========================================


In [42]:
# A. PREPARE DATA
print("Loading data...")
# NOTE: You must provide the correct path to your JSON dataset
df = pd.read_json('nexus_elite_dataset_v2.json') 

# OOD Split (Unique ID based)
unique_ids = df['original_id'].unique()
np.random.shuffle(unique_ids)
split_idx = int(len(unique_ids) * config.OOD_TEST_RATIO)
ood_ids = unique_ids[:split_idx]

train_df_full = df[~df['original_id'].isin(ood_ids)].reset_index(drop=True)
ood_df = df[df['original_id'].isin(ood_ids)].reset_index(drop=True)

# Train/Val Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(train_df_full, groups=train_df_full['original_id']))
train_df = train_df_full.iloc[train_idx].reset_index(drop=True)
val_df = train_df_full.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | OOD: {len(ood_df)}")

# B. MODEL SETUP
print("\nInitializing Backbone & LoRA...")
model = NexusV2Production(config.BACKBONE_MODEL).to(config.DEVICE)
if config.USE_LORA_STAGE1:
    model = apply_lora_final(model, config.LORA_R)

# C. STAGE 1: TRIPLET TRAINING
print(f"\n=== STAGE 1: CAUSAL TRIPLET TRAINING ({config.STAGE1_EPOCHS} Epochs) ===")
triplet_ds = NexusTripletDataset(train_df)
triplet_loader = DataLoader(triplet_ds, batch_size=16, shuffle=True) # Batch size 16 for triplets

trainer_s1 = CausalTripletTrainer(model, triplet_loader)
for epoch in range(config.STAGE1_EPOCHS):
    loss = trainer_s1.train_epoch()
    print(f"Epoch {epoch+1}: Triplet Loss = {loss:.4f}")
    
torch.save(model.state_dict(), "nexus_stage1_backbone.pt")



Loading data...
Train: 10759 | Val: 2130 | OOD: 767

Initializing Backbone & LoRA...


c:\Users\alper\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


trainable params: 1,622,016 || all params: 185,453,568 || trainable%: 0.8746

=== STAGE 1: CAUSAL TRIPLET TRAINING (3 Epochs) ===
✓ Created 4040 Causal Triplets from 6871 unique news events.


Stage 1: Causal Triplet Training:   0%|          | 1/253 [01:39<6:59:09, 99.80s/it, triplet_loss=0.2999]


KeyboardInterrupt: 

In [ ]:
# D. STAGE 2: MULTI-HEAD TRAINING
print(f"\n=== STAGE 2: MULTI-HEAD TRAINING ({config.STAGE2_EPOCHS} Epochs) ===")

# Freeze logic
if config.FREEZE_LORA_STAGE2:
    print("Freezing Encoder (including LoRA)...")
    for param in model.encoder.parameters():
        param.requires_grad = False

train_ds = NexusDatasetV2(train_df)
val_ds = NexusDatasetV2(val_df)

# Weighted Sampler
train_labels = (train_df['label'] != 0).astype(int).values
class_counts = np.bincount(train_labels)
class_weights = np.sqrt(1.0 / class_counts)
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False)

optimizer = get_nexus_optimizer(model)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=len(train_loader), num_training_steps=len(train_loader)*config.STAGE2_EPOCHS)

best_val_loss = float('inf')

for epoch in range(config.STAGE2_EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for batch in pbar:
        texts, gate_gt, dir_gt, tp_gt, val_gt = batch
        
        gate_gt = gate_gt.view(-1, 1).to(config.DEVICE)
        dir_gt = dir_gt.view(-1, 1).to(config.DEVICE)
        tp_gt = tp_gt.view(-1, 1).to(config.DEVICE)
        val_gt = val_gt.view(-1, 1).to(config.DEVICE)
        
        inputs = model.tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=config.MAX_LENGTH).to(config.DEVICE)
        outputs = model.forward(inputs['input_ids'], inputs['attention_mask'])
        
        loss, loss_dict = nexus_combined_loss_v7_final(outputs, (gate_gt, dir_gt, tp_gt, val_gt))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    # Validation
    model.eval()
    val_loss = 0
    all_gate_gts = []
    all_gate_preds = []
    
    with torch.no_grad():
        for batch in val_loader:
            texts, gate_gt, dir_gt, tp_gt, val_gt = batch
            gate_gt = gate_gt.view(-1, 1).to(config.DEVICE)
            dir_gt = dir_gt.view(-1, 1).to(config.DEVICE)
            tp_gt = tp_gt.view(-1, 1).to(config.DEVICE)
            val_gt = val_gt.view(-1, 1).to(config.DEVICE)
            
            inputs = model.tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True, max_length=config.MAX_LENGTH).to(config.DEVICE)
            outputs = model.forward(inputs['input_ids'], inputs['attention_mask'])
            
            loss, _ = nexus_combined_loss_v7_final(outputs, (gate_gt, dir_gt, tp_gt, val_gt))
            val_loss += loss.item()
            
            all_gate_gts.extend(gate_gt.cpu().numpy())
            all_gate_preds.extend(torch.sigmoid(outputs['gate']).cpu().numpy() > 0.5)

    avg_val = val_loss / len(val_loader)
    gate_f1 = precision_recall_fscore_support(all_gate_gts, all_gate_preds, average='binary', zero_division=0)[2]
    print(f"Val Loss: {avg_val:.4f} | Gate F1: {gate_f1:.4f}")
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "nexus_production_final.pt")
        print("✓ Saved Best Model")

# E. CALIBRATION & TESTING
model.load_state_dict(torch.load("nexus_production_final.pt"))
optimal_t = calibrate_model(model, val_loader, config.DEVICE)
counterfactual_test(model)
print("\n✓ Training Complete. Model saved as 'nexus_production_final.pt'")

